# Robustness

**One crash costs you a lap.**

Unit 10's agent steers by MPC and plans its speed with `MU_FRACTION = 0.58` of the grip Unit 8 measured.
It is clean on all five training tracks and on the unseen `test-circuit-seed2026`.
But every one of those races was on **one** car: racekit's carla-like car, the car Unit 8 measured.
The car you will race is not that car.
Unit 8's numbers are estimates, and CARLA is not racekit: the real grip, the real steering gain, the real throttle may each be a few per cent off.

racekit can build such cars on purpose.
`carla_like(noise=True, perturb=k)` is car `k` of a seeded **family**: the carla-like car with its grip, steering gain, throttle, brake, mass and drag each changed by a few per cent (§3 says exactly how).
Here is Unit 10's agent, exactly as shipped (0.58), on four cars of that family, on `test-circuit` (race seed k mod 5, a time limit of 480 s):

In [ ]:
import inspect
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import PERTURBATION_FAMILY, carla_like, perturbation_factors
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_09, UNIT_10, UNIT_11 = AGENTS / "unit-09.py", AGENTS / "unit-10.py", AGENTS / "unit-11.py"
workdir = Path(mkdtemp(prefix="roar-unit-11-"))
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
HOLD_S = 1.0  # s: the respawn hold, 20 ticks without calling the agent
NO_TIMING = {"MPC_BUDGET_MS": None}  # every MPC race here: no wall clock inside the race (Unit 10, section 7)
HOOK_CAP = 480.0  # s: the hook's time limit


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def race(source, k=None, track="test-circuit", seed=None, lag_ticks=None, max_seconds=450.0):
    # One race on car k of the perturbed family (k = None: the unperturbed car), with its noise.
    # The race seed is k mod 5 unless one is given (0 for the unperturbed car).
    if seed is None:
        seed = 0 if k is None else k % 5
    car = carla_like(noise=True, perturb=k, lag_ticks=lag_ticks)
    return run_race(source, track, car=car, seed=seed, max_seconds=max_seconds)


# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


# Unit 9's line-relative clean rule, against test-circuit's line as the agents ship it.
with shim_installed():
    Unit11, _ = load_submission(UNIT_11)
track = load_track("test-circuit")
line_offsets = np.array(Unit11(track.waypoints, None).offsets)


def line_error(result, offsets=None):
    # The largest |cross-track error from the race's own line| over the driving rows (held == 0).
    offsets = line_offsets if offsets is None else offsets
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets=None):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def summary(result, offsets=None):
    verdict = "clean" if clean(result, offsets) else "NOT clean"
    finished = "finished" if result.finished else "did not finish"
    crashes = ", ".join(f"{t:.2f}" for t in result.respawn_times)
    plural = "" if result.collisions == 1 else "s"
    return (f"{verdict}: {result.total_time:.2f} s, {result.collisions} collision{plural}"
            f"{' (at ' + crashes + ' s)' if crashes else ''}, {finished}")


unit10_058 = variant(UNIT_10, "u10-0.58", **NO_TIMING)  # Unit 10 as shipped (0.58), timing off
HOOK_CARS = (1, 8, 16, 20)
hook = {k: race(unit10_058, k, max_seconds=HOOK_CAP) for k in HOOK_CARS}
for k, result in hook.items():
    print(f"car {k:2d} (race seed {k % 5}): {summary(result)}")
assert clean(hook[1]) and abs(hook[1].total_time - 275.30) <= TICK
assert hook[8].finished and hook[8].collisions == 1 and abs(hook[8].total_time - 471.85) <= TICK  # (plan q9)
for k in (16, 20):
    assert not hook[k].finished and hook[k].collisions >= 2  # crash loops (U11.2 at a 700 s cap: 4 each)

fig, ax = plt.subplots(figsize=(11, 3.5))
for k, result in hook.items():
    ax.plot(result.telemetry["t"], result.telemetry["progress"], label=f"car {k}")
ax.axhline(hook[1].progress_required, color="grey", linestyle="--", label="the finish")
ax.set_xlabel("t (s)")
ax.set_ylabel("progress (waypoints)")
ax.set_title("Unit 10 (f = 0.58) on four perturbed cars: every crash sends progress back to 0")
ax.legend()
plt.show()

Car 1 is clean in 275.30 s.
Car 8 spins at 195.85 s, and its progress falls to 0: the car is back at the start, and it finishes at 471.85 s, almost 200 s after car 1.
Cars 16 and 20 crash, start again, crash again, and are still racing when the time runs out; the book's tests, with a 700 s limit, see them crash 4 times each.
Over all 24 cars the book's tests race (plan `roar-010`, premise U11.2), 21 are clean, car 8 crashes once, and cars 16 and 20 never finish.

"One crash costs you a lap" is an understatement.
In this race a crash **restarts the race**: the car goes back to the start, and the clock keeps running.
And an agent that crashed once, at the same speed plan, can crash again, and again.

This unit measures what a crash costs (§1) and what to optimise when the car is uncertain (§2), builds a family of cars to test on (§3), re-tunes the fraction on it and checks it on cars it never saw (§4), detects trouble (§5), and survives it (§6).
§7 is the submission, `agents/unit-11.py`.

## 1. The cost of a crash

The race rules (design 013 §4.2, as the official runner does it): on a major collision (leaving the drivable width, or a spin-out) the car is put back **at the start pose, at rest**, held there for 20 ticks (1 s) without calling the agent, and its progress is set back to **0**.
The clock is not reset.

So a crash at time t throws away everything driven so far, adds the 1 s hold, and then the whole race has to be driven again, from rest, exactly like the start.
If the race after the restart takes as long as a fresh race, T, the race time is t + 1 s + T: **a crash at time t costs t + 1 s.**
Read it off car 8's race:

In [ ]:
crashed = hook[8]
telemetry = crashed.telemetry
t_crash = crashed.respawn_times[0]
held = telemetry[telemetry["held"] == 1]
after = telemetry[(telemetry["t"] > t_crash) & (telemetry["held"] == 0)]
print(f"car 8: crash at t = {t_crash:.2f} s ({crashed.respawn_causes[0]}), at progress "
      f"{telemetry['progress'][telemetry['t'] < t_crash][-1]} of the {crashed.progress_required} needed to finish")
print(f"held: {len(held)} rows, from {held['t'][0]:.2f} to {held['t'][-1]:.2f} s, speed {np.max(held['v']):.1f} m/s")
print(f"the first row after the hold: t = {after['t'][0]:.2f} s, progress {after['progress'][0]}")
remainder = crashed.total_time - t_crash - HOLD_S
print(f"the race after the restart: {crashed.total_time:.2f} - {t_crash:.2f} - {HOLD_S} = {remainder:.2f} s "
      f"(car 1's whole race: {hook[1].total_time:.2f} s)")
print(f"the crash cost about t + 1 s = {t_crash + HOLD_S:.2f} s")
assert abs(t_crash - 195.85) <= TICK and len(held) == 20 and np.max(held["v"]) == 0.0 and after["progress"][0] == 0
assert abs(remainder - 275.00) <= TICK

Car 8 had driven more than two of its three laps.
The hold is 20 rows at rest, the next row is at progress 0, and then the car drives the whole race again: 275.00 s, a normal race on this family (car 1's took 275.30 s).
Car 8 lost the 195.85 s it had driven, plus the hold: **196.85 s, 70% of a race**, for one spin.
A crash late in the race costs more than a crash early in it; a crash on the last corner costs a whole race.

**Is the restart exactly a fresh race?**
Almost.
The noise is not re-seeded, so the restart is a different draw of it; and the agent is not reset: racekit's runner, like the official one, resets the car and the progress, never the agent.
In particular the agent's speed loop keeps the integral it had built up at the moment of the crash, so the restarted car takes a little more throttle at first than a fresh one would, and finishes a little earlier.
The book's tests (premise U11.3) raced, for each of the 7 cars that crash exactly once with this unit's agent, a **fresh** race of the same car in the same configuration: the race after the restart was 0.40 to 0.65 s shorter every time.
So the rule is: **a crash at time t costs t + 1 s, to within a second.**
§6 measures one of these yourself.

**The crash loop.**
The restart is a new race, and a new race can crash too:

In [ ]:
for k in (16, 20):
    times = hook[k].respawn_times
    starts = [0.0] + [t + HOLD_S for t in times]
    attempts = ", ".join(f"{t - s:.2f} s" for s, t in zip(starts, times))
    print(f"car {k}: crashes at {', '.join(f'{t:.2f}' for t in times)} s; the attempts lasted {attempts}, "
          f"then the {HOOK_CAP:.0f} s limit")
assert [round(t, 2) for t in hook[16].respawn_times] == [190.60, 291.90]
assert [round(t, 2) for t in hook[20].respawn_times] == [99.10, 294.65]

Car 16's first attempt lasts 190.60 s, its second 100.30 s; car 20's 99.10 s, then 194.55 s.
Every attempt ends in a crash, so neither car ever finishes: a **crash loop**, and in a real race a did-not-finish (DNF).
The agent has no idea anything happened: it drives the restart exactly as it drove the first attempt, with the same speed plan, on the same car that just crashed under it.
§6 changes that.

## 2. Expected race time over a family

You do not know which car you will race.
Treat the real car as one draw from a family of cars that might be it, and ask: which agent has the shortest race time **on average over the family**?
That is the **expected race time**: the mean of the race times over the family, with a race that did not finish counted at the time limit.

Counting a DNF at the limit is a choice, and a gentle one: in a real race a DNF earns no time at all.
It makes the mean depend on the limit, so two fractions are compared at the **same** limit.
Here are the book's tests (premise U11.2 and U11.4): Unit 10's agent at four fractions on the 24 training cars of the family (race seed k mod 5, a 700 s limit), with every race that was not clean:

In [ ]:
CAP = 700.0  # s: the book's tests' time limit
FAMILY_SIZE = 24
# plan roar-010, premise U11.2/U11.4: Unit 10 on training cars 0-23. The mean race time (a DNF counted at the cap),
# and every race that was not clean: (car, race time, finished).
PREMISE_MEAN = {0.58: 318.66, 0.56: 294.35, 0.54: 279.95, 0.52: 282.58}
PREMISE_NOT_CLEAN = {
    0.58: [(8, 471.85, True), (16, 700.0, False), (20, 700.0, False)],
    0.56: [(8, 474.00, True), (9, 486.90, True)],
    0.54: [],
    0.52: [],
}
clean_mean = {}
for f, mean in PREMISE_MEAN.items():
    bad = PREMISE_NOT_CLEAN[f]
    clean_mean[f] = (FAMILY_SIZE * mean - sum(time for _, time, _ in bad)) / (FAMILY_SIZE - len(bad))
    print(f"f = {f}: {FAMILY_SIZE - len(bad):2d}/24 clean, their mean {clean_mean[f]:.2f} s; "
          f"expected race time {mean:.2f} s")
best = min(PREMISE_MEAN, key=PREMISE_MEAN.get)
print("the fraction with the shortest expected race time:", best)
assert best == 0.54  # U11.4

gain = clean_mean[0.54] - clean_mean[0.56]
cost = 474.00 - clean_mean[0.56]
print(f"one step up (0.54 -> 0.56) makes a clean race {gain:.2f} s faster; car 8's crash at 0.56 cost {cost:.1f} s")
print(f"it pays only if fewer than {gain / cost:.1%} of the races crash: one in {cost / gain:.0f}")

small = [hook[k].total_time if hook[k].finished else HOOK_CAP for k in HOOK_CARS]
print(f"the hook's four cars at 0.58 (a DNF at {HOOK_CAP:.0f} s): mean {np.mean(small):.2f} s")

fig, ax = plt.subplots(figsize=(7, 3.5))
fractions = sorted(PREMISE_MEAN)
ax.plot(fractions, [PREMISE_MEAN[f] for f in fractions], "o-", label="expected race time (DNF at 700 s)")
ax.plot(fractions, [clean_mean[f] for f in fractions], "s--", label="mean of the clean races")
ax.set_xlabel("MU_FRACTION")
ax.set_ylabel("race time (s)")
ax.set_title("Unit 10 on the 24 training cars (the book's tests)")
ax.legend()
plt.show()

The clean races get faster as the fraction rises, about 2.5 s per step of 0.02.
The expected race time does not: it is lowest at **0.54**, because above it a few crashes cost far more than every clean race gains.
One step up, from 0.54 to 0.56, saves 2.52 s on a clean race; car 8's crash at 0.56 cost 196.6 s.
The faster fraction pays only if fewer than about 1.3% of races crash, one race in 78.
And 24 cars cannot show a crash rate that small: one crash in 24 is already 4%.
That is why a fraction is never chosen at the edge of what was raced clean (§4).

The hook's four cars are a poor sample of the family: chosen for their crashes, their mean at 0.58 is 426.79 s.
A family mean needs the whole family, and that is what the book's tests race.

## 3. The perturbed family

`carla_like(noise=True, perturb=k)` is car `k` of racekit's **perturbed family**.
Each of six properties of the carla-like car is multiplied by a factor drawn for car `k`:

| factor | range |
|---|---|
| grip (every grip section's μ) | ×0.92 to ×1.00 |
| steering gain (the full-lock wheel angle) | ×0.95 to ×1.05 |
| throttle authority | ×0.90 to ×1.10 |
| brake authority | ×0.90 to ×1.05 |
| mass and yaw inertia (one factor) | ×0.95 to ×1.10 |
| drag | ×0.90 to ×1.10 |

`perturbation_factors(k)` returns car `k`'s factors, and `PERTURBATION_FAMILY` holds the ranges.
Each factor is drawn from a random generator seeded by `k`, so car `k` is the same car on every machine, and `perturb=None` (the default) is the unperturbed car exactly.
The factors are **relative** (1.0 is the car Unit 8 measured): they tell you how car `k` differs, not what the car is.

These ranges are **the book's model of model error, not a measurement of CARLA**.
Nobody knows how far CARLA's car is from racekit's, or from your own Unit 8 measurement of it.
The ranges say: the grip may be up to 8% lower than measured, never higher (the dangerous direction: more grip than planned for only costs time, less of it crashes), and everything else a few per cent either way.
The actuator lag is not in the family: Unit 8 identifies it as a whole number of ticks, and §6 shows what one tick of error does.

**Training and held-out cars.**
Cars 0 to 23 are the **training** cars: tune on them.
Cars 24 to 47 are **held out**: race them only to check a setting that is already chosen, exactly as Checkpoint A kept an unseen track.
Each car races with race seed k mod 5, so the noise varies across the family too.

In [ ]:
print("PERTURBATION_FAMILY:", dict(PERTURBATION_FAMILY))
factors = {k: perturbation_factors(k) for k in range(48)}
assert perturbation_factors(8) == factors[8]  # seeded by k: the same car every time
TRAINING, HELD_OUT = tuple(range(24)), tuple(range(24, 48))
for k in HOOK_CARS:
    print(f"car {k:2d}: " + ", ".join(f"{name} x{value:.3f}" for name, value in factors[k].items()))

PREMISE_CRASHED_058 = (8, 16, 20)  # not clean at 0.58 in the book's tests (U11.2)
by_grip = sorted(TRAINING, key=lambda k: factors[k]["grip"])
print("the six lowest-grip training cars:", by_grip[:6])
for k in by_grip[:6]:
    effective = 0.58 / factors[k]["grip"]
    print(f"  car {k:2d}: grip x{factors[k]['grip']:.3f}, throttle x{factors[k]['throttle']:.3f}: "
          f"f = 0.58 of the measured grip is {effective:.3f} of this car's"
          f"{'  (crashed at 0.58)' if k in PREMISE_CRASHED_058 else ''}")
assert by_grip[:6] == [16, 18, 8, 9, 20, 13]

fig, ax = plt.subplots(figsize=(7, 4))
others = [k for k in TRAINING if k not in PREMISE_CRASHED_058]
ax.scatter([factors[k]["grip"] for k in others], [factors[k]["throttle"] for k in others], label="clean at 0.58")
ax.scatter([factors[k]["grip"] for k in PREMISE_CRASHED_058], [factors[k]["throttle"] for k in PREMISE_CRASHED_058],
           marker="x", s=80, label="not clean at 0.58")
ax.set_xlabel("grip factor")
ax.set_ylabel("throttle factor")
ax.set_title("the 24 training cars, and Unit 10's crashes at 0.58 (the book's tests)")
ax.legend()
plt.show()

The three cars that crash at 0.58 are all among the five lowest-grip training cars.
The planner uses 0.58 of the grip Unit 8 measured, but car 16 has only 0.930 of that grip, so on car 16 the planner really uses 0.624 of the grip that is there: above 0.60, MPC's family edge in Unit 10.
Grip is not the whole story, though: car 18 has almost as little grip (0.934) and is clean, and its throttle is 9% weak, so it arrives at every corner a little slower.
A single factor does not decide a crash; the only test is to race the car.

## 4. The family edge, the margin rule, and cars never tuned on

Units 7 to 10 chose a fraction by the **margin rule**: find the highest fraction that is clean on every member of the training family (the **edge**), then ship one grid step inside it.
On the perturbed family, the probe record of plan `roar-010` (race seed 0 on every car) found the edge at **0.56**, so the rule gives **0.54**.
With the race seeds k mod 5 that the book's tests use, 0.56 crashes twice (cars 8 and 9, §2): a different draw of the noise moved the edge down by a step.
That is exactly what the step of margin is for, and 0.54 is clean on all 24 training cars either way.
It also has the shortest expected race time (§2), so both rules agree: this unit's agent, `agents/unit-11.py`, ships **`MU_FRACTION = 0.54`**.

**The held-out check.**
The fraction was chosen on cars 0 to 23.
The test is the cars it never saw.
Here is `agents/unit-11.py` (0.54; with no crash it races exactly like Unit 10's file at 0.54, §7) on two held-out cars, 24 and 32, two of the three lowest-grip cars of the held-out half:

In [ ]:
unit11 = variant(UNIT_11, "u11", **NO_TIMING)  # the shipped file (0.54), timing off
held_out = {k: race(unit11, k) for k in (24, 32)}
for k, result in held_out.items():
    print(f"held-out car {k} (grip x{factors[k]['grip']:.3f}, race seed {k % 5}): {summary(result)}, "
          f"{line_error(result):.2f} m from its line")
assert clean(held_out[24]) and abs(held_out[24].total_time - 280.20) <= TICK
assert clean(held_out[32]) and abs(held_out[32].total_time - 280.65) <= TICK

Both are clean, in 280.20 and 280.65 s.
The book's tests race all 24 held-out cars (premise U11.6, the unit's gate): **24 of 24 are clean** at 0.54, median 279.95 s, never more than 1.35 m from the line.
At 0.58, 4 of those 24 crash (premise U11.5).

**A margin bought on one track.**
Everything so far was on `test-circuit`.
Take the same agent, at the same 0.54, to `test-circuit-seed2026` (Checkpoint A's unseen track, with its own line and its own measured grip written in, as in Unit 10), on car 6 of the family:

In [ ]:
MEASURED_MU_G_2026 = (7.70, 7.69, 7.70, 9.81, 8.16, 8.04)  # Unit 8's grip probes on test-circuit-seed2026 (m/s^2)
seed2026 = load_track("test-circuit-seed2026")
seed2026_control = solve_line(seed2026)
seed2026_offsets = spline_basis(len(seed2026)) @ seed2026_control
unit11_2026 = variant(UNIT_11, "u11-seed2026", LINE_CONTROL=tuple(float(value) for value in seed2026_control),
                      MU_G=MEASURED_MU_G_2026, **NO_TIMING)
transfer = race(unit11_2026, 6, track="test-circuit-seed2026", seed=0, max_seconds=300.0)
print(f"test-circuit-seed2026, car 6 (grip x{factors[6]['grip']:.3f}), f = 0.54, seed 0: "
      f"{summary(transfer, seed2026_offsets)}")
assert not clean(transfer, seed2026_offsets)  # U11.8: car 6 is one of the cars that crash
assert transfer.collisions == 1 and abs(transfer.respawn_times[0] - 129.40) <= TICK

Car 6 crashes at 129.40 s (time limited to 300 s here; the book's tests let it finish, in 422.95 s).
In the book's tests (premise U11.8: cars 0 to 11, race seed 0) 4 of the 12 crash on `test-circuit-seed2026` at 0.54 (cars 6, 8, 9 and 10), while all 48 perturbed `test-circuit` races at 0.54 are clean.
**A margin bought on one track does not carry over to another.**
The rule was right about the family on the track it was raced on, and it said nothing about another track.
This is a finding, not a fix: Checkpoint B asks for an agent that completes perturbed races on tracks it was not tuned on, and Challenge 10 looks for a fraction that is clean here.

## 5. Detecting trouble

A margin makes crashes rare; it cannot make them impossible.
So the agent should notice when one happens, and perhaps before.
It has its sensors and nothing else: nobody tells it that it crashed.

**The respawn signal.**
In CARLA a collision is reported by the **collision sensor**, as an impulse (N·s); the shim gives it to the agent as `collision_sensor.get_last_gym_observation()`, a 3-vector.
racekit reports a large impulse on the first step after the respawn hold, on that one step only.
CARLA's sensor may **repeat its last reading** on later steps.
So a respawn is detected **on an edge**, not on a level:

- the impulse's norm is over `RESPAWN_IMPULSE` = 100 N·s,
- the reading is **not the previous step's** (a repeated reading is the same event), and
- no respawn was handled in the last `REFRACTORY_TICKS` = 40 steps (2 s): a reading that flickers back and forth is still one event, and two real respawns cannot come 2 s apart (the hold alone takes 1 s, and then the car has to reach a wall from rest).

Here is the rule on scripted readings (no race), next to a detector that counts every step over the threshold:

In [ ]:
QUIET, CRASH = (0.0, 0.0, 0.0), (0.0, -1000.0, 0.0)  # N s: racekit's impulse after a major collision


def detect(readings, threshold=100.0, refractory=40):
    # The respawn rule of agents/unit-11.py on one collision reading per step (steps counted from 1):
    # over the threshold, not the previous step's reading, and none handled in the last `refractory` steps.
    handled, previous, last = [], None, None
    for step, reading in enumerate(readings, start=1):
        edge = float(np.linalg.norm(reading)) > threshold and reading != previous
        if edge and (last is None or step - last >= refractory):
            handled.append(step)
            last = step
        previous = reading
    return handled


scripts = {
    "racekit: one step": [QUIET] * 5 + [CRASH] + [QUIET] * 60,
    "repeated for 10 steps": [QUIET] * 5 + [CRASH] * 10 + [QUIET] * 60,
    "flickers within 2 s": [QUIET] * 5 + [CRASH, QUIET, CRASH, QUIET, CRASH] + [QUIET] * 60,
    "two respawns 3 s apart": [QUIET] * 5 + [CRASH] + [QUIET] * 59 + [CRASH] + [QUIET] * 10,
}
for name, readings in scripts.items():
    level = sum(1 for reading in readings if float(np.linalg.norm(reading)) > 100.0)
    print(f"{name:24}: edge-triggered handles steps {detect(readings)}; level-triggered counts {level}")
assert [len(detect(readings)) for readings in scripts.values()] == [1, 1, 1, 2]

print(inspect.getsource(Unit11.respawned))

A level-triggered detector would see 10 crashes in one repeated reading and 3 in a flicker; the edge rule with its 2 s window sees one, and still sees two real respawns 3 s apart.
The shipped method, printed above, is the same rule.

**Warnings before a crash.**
Can the agent see a crash coming?
Two instruments, both from sensors the agent already reads:

- the **sideslip** β, the angle between where the car is going and where it is pointing: β = atan2(v_y, v_x) − ψ (the velocity sensor's direction minus the yaw, wrapped).
  Unit 6's spin-out is |β| beyond a limit; a car on its line has a small β.
- the **line error**, the agent's own cross-track error from its line (Unit 2, on Unit 9's line); a clean race stays under 2 m.

`agents/unit-11.py` computes both every step, above `WARN_MIN_SPEED` = 5 m/s, and counts a **warning** when |β| > `WARN_BETA_DEG` = 8° or |line error| > `WARN_ERROR_M` = 1.6 m.
How early do they warn, and do they warn when nothing is wrong?
Record them in a subclass, on one clean race and before two crashes (Unit 11's file at Unit 10's 0.58: with no crash it races exactly like Unit 10's, so these are the hook's races):

In [ ]:
def logged(source):
    # The submission's class, recording after every step its sideslip (deg), its line error (m) and its speed (m/s),
    # and keeping the agent, to read its counters after the race.
    with shim_installed():
        Agent, _ = load_submission(source)
    log = {"beta": [], "error": [], "speed": [], "agent": []}

    class Logged(Agent):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            log["agent"].append(self)

        async def step(self):
            await super().step()
            log["beta"].append(self.beta)
            log["error"].append(self.cross_track_error)
            log["speed"].append(float(np.linalg.norm(self.velocity_sensor.get_last_gym_observation())))

    return Logged, log


def lead_time(t, values, threshold, speed, t_crash):
    # How long before the crash the last run of steps over the threshold (above 5 m/s) began; None if never over it.
    over = (np.abs(values) > threshold) & (speed > 5.0) & (t < t_crash)
    if np.sum(over) == 0:
        return None
    start = int(np.nonzero(over)[0][-1])
    while start > 0 and over[start - 1]:
        start -= 1
    return float(t_crash - t[start])


unit11_058 = variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)
logs = {}
for k, cap in ((1, 450.0), (16, 195.0), (20, 105.0)):  # the crashes are at 190.60 and 99.10 s
    Logged, log = logged(unit11_058)
    result = race(Logged, k, max_seconds=cap)
    t = result.telemetry["t"][result.telemetry["held"] == 0]  # one telemetry row per step() call
    logs[k] = (result, t, np.array(log["beta"]), np.array(log["error"]), np.array(log["speed"]))

result, t, beta, error, speed = logs[1]
moving = speed > 5.0
peak_beta, peak_error = np.max(np.abs(beta[moving])), np.max(np.abs(error[moving]))
print(f"car 1, clean ({result.total_time:.2f} s): the largest |beta| {peak_beta:.2f} deg, "
      f"the largest |line error| {peak_error:.2f} m")
assert clean(result) and abs(result.total_time - 275.30) <= TICK
assert abs(peak_beta - 5.23) <= 0.01 and abs(peak_error - 1.30) <= 0.01
leads = {}
for k in (16, 20):
    result, t, beta, error, speed = logs[k]
    t_crash = result.respawn_times[0]
    leads[k] = (lead_time(t, beta, 8.0, speed, t_crash), lead_time(t, error, 1.6, speed, t_crash))
    print(f"car {k}: crash at {t_crash:.2f} s; |beta| > 8 deg {leads[k][0]:.2f} s before it; "
          f"|line error| > 1.6 m: {'%.2f s before it' % leads[k][1] if leads[k][1] is not None else 'never'}")
assert abs(leads[16][0] - 1.10) <= TICK and abs(leads[16][1] - 0.25) <= TICK
assert abs(leads[20][0] - 0.80) <= TICK and leads[20][1] is None

fig, (ax_b, ax_e) = plt.subplots(1, 2, figsize=(11, 3.5))
for k in (16, 20):
    result, t, beta, error, speed = logs[k]
    window = t > result.respawn_times[0] - 6.0
    ax_b.plot(t[window] - result.respawn_times[0], beta[window], label=f"car {k}")
    ax_e.plot(t[window] - result.respawn_times[0], error[window], label=f"car {k}")
for ax, limit, peak, unit in ((ax_b, 8.0, peak_beta, "deg"), (ax_e, 1.6, peak_error, "m")):
    for sign in (1, -1):
        ax.axhline(sign * limit, color="grey", linestyle="--")
        ax.axhline(sign * peak, color="green", linewidth=0.8)
    ax.set_xlabel("time to the crash (s)")
    ax.legend()
ax_b.set_ylabel("sideslip beta (deg)")
ax_b.set_title("beta: dashed the 8 deg warning, green the clean race's peak")
ax_e.set_ylabel("line error (m, left +)")
ax_e.set_title("line error: dashed the 1.6 m warning")
fig.tight_layout()
plt.show()

In the clean race |β| never goes past 5.23° and the line error past 1.30 m, so neither warning fires once: no false alarms here.
Before a crash, β warns first: 1.10 s before car 16's crash and 0.80 s before car 20's (the probe record of plan `roar-010`, over more crashes: 0.9 to 2.4 s).
The line error warns late, 0.25 s before car 16's crash (probe record: 0.15 to 0.9 s), and **not at all** before car 20's: car 20 spins on its line.
A warning about a second ahead is not much, at 30 m/s.

**The honest negative result.**
The natural next step is to act on the warning: fall back (§6's fallback: pure pursuit, a lower speed plan) as soon as |β| > 8°, before the crash.
The book's tests did exactly that (premise U11.9, `WARN_FALLBACK = True`, the 24 training cars at 0.58): every car that crashed without it **still crashed**, each about 11 s later than without the warning, for a mean race time of **298.07 s** against **296.66 s** for falling back on the respawn alone.
The probe record found the same at a 6° threshold.
No clean car was tripped; the warning came, the fallback was taken, and it was too late to save the car.
That result is for **these thresholds and this reaction**, not for every possible one: Challenge 9 tries braking instead.
So the shipped agent **counts** warnings, as an instrument, and acts on the respawn alone.
§6 races the warning trigger on car 20.

## 6. The fallback

After a respawn the car is at the start, at rest, and the race begins again.
The agent is not: it is still carrying the moment of the crash.
`fall_back()` puts it right, and makes it more careful:

1. **Reset the MPC's memory.** The lag queue and `mpc_last` hold the commands from the crash, and the plan is a plan for a corner the car is no longer in: clear them (`mpc_queue`, `mpc_last`, `plan`).
2. **Re-anchor, one step later.** The waypoint index still points at the crash; it must jump back to the start (`nearest_index`, Unit 2).
   But on CARLA the first step after a respawn can still carry the **stale crash pose**, so the agent re-anchors on the **next** step, not this one.
3. **Degrade.** The car that crashed under this plan is not the car the plan was made for.
   Switch the steering to `FALLBACK_STEERING = "pursuit"`, and multiply the lateral limit by `FALLBACK_SCALE = 0.86`, re-planning the speed profile (Unit 7's forward–backward pass) with it; at a second respawn, multiply again.
   At 0.58 the restart plans with 0.58 × 0.86 ≈ 0.50 of the measured grip.

In [ ]:
print(inspect.getsource(Unit11.fall_back))

Logged, log = logged(unit11_058)
saved = race(Logged, 20)
(saved_agent,) = log["agent"]
twin = race(variant(UNIT_11, "u11-twin", STEERING="pursuit", MU_FRACTION=0.58 * 0.86, **NO_TIMING), 20)
remainder = saved.total_time - saved.respawn_times[0] - HOLD_S
print(f"car 20, Unit 11 at 0.58: {summary(saved)}; respawns handled {saved_agent.respawns}, "
      f"mode {saved_agent.mode!r}, limit_scale {saved_agent.limit_scale}, warnings counted {saved_agent.warnings}")
t = saved.telemetry["t"][saved.telemetry["held"] == 0]
beta, error, speed = np.array(log["beta"]), np.array(log["error"]), np.array(log["speed"])
sliding = (np.abs(beta) > 8.0) & (speed > 5.0)
off_line = (np.abs(error) > 1.6) & (speed > 5.0)
before = t < saved.respawn_times[0]
print(f"   ... warning steps before the crash: {np.sum((sliding | off_line) & before)} (all sideslip); after the restart: "
      f"{np.sum((sliding | off_line) & ~before)} ({np.sum(sliding & ~before)} sideslip, {np.sum(off_line & ~before)} line error), "
      f"the largest line error after it {np.max(np.abs(error[~before])):.2f} m")
print(f"car 20, a fresh race in the fallback configuration (pursuit, f = 0.58 x 0.86): {summary(twin)}")
print(f"the race after the restart: {remainder:.2f} s, {remainder - twin.total_time:+.2f} s against the fresh race")
assert saved.finished and saved.collisions == 1 and abs(saved.total_time - 385.05) <= TICK  # U11.5
assert saved_agent.respawns == 1 and saved_agent.mode == "fallback" and saved_agent.warnings == 121
assert np.sum((sliding | off_line) & before) == np.sum(sliding & before) == 16
assert np.sum(off_line & ~before) == 104 and abs(np.max(np.abs(error[~before])) - 2.04) <= 0.01
assert clean(twin) and abs(twin.total_time - 285.35) <= TICK
assert abs(remainder - twin.total_time) <= 1.0  # U11.3: within a second (here 0.40 s shorter)

Where Unit 10 crash-looped on car 20 (the hook), Unit 11's file crashes once, at 99.10 s, falls back, and finishes in 385.05 s.
The race after the restart took 284.95 s, 0.40 s less than a fresh race of car 20 in the fallback configuration (285.35 s): the restart is a fresh race, to within a second, as §1 said, and the speed loop's leftover integral makes it the shorter.
In the book's tests (premise U11.5) every one of the 48 perturbed races at 0.58 finishes with at most one collision with the fallback.
Of its 121 warnings, 16 came in the 0.8 s before the spin, all sideslip; the other 105 came after the restart, 104 of them from the line error: pure pursuit follows the line less closely than MPC (Unit 10), and at 0.50 of the grip it drifts up to 2.04 m from it in a few corners.
The warnings are counted, never acted on.

**Why a *different* controller?**
Re-planning slower would help a car whose only problem is grip.
But some model errors are not about grip, and no fraction fixes them.
Take the car whose actuator lag is **the lag Unit 8 identified, plus one tick**: `carla_like(noise=True, lag_ticks=2)`, otherwise the unperturbed car.
Unit 10's MPC at 0.54, then Unit 9's pure pursuit at 0.54 (`agents/unit-09.py` as shipped), then this unit's file:

In [ ]:
lag2_mpc = race(variant(UNIT_10, "u10-0.54", MU_FRACTION=0.54, **NO_TIMING), lag_ticks=2, max_seconds=200.0)
telemetry = lag2_mpc.telemetry
where = [int(telemetry["progress"][telemetry["t"] < t_c][-1]) for t_c in lag2_mpc.respawn_times]
print(f"lag 2, Unit 10 (MPC, f = 0.54): {summary(lag2_mpc)}")
print(f"   ... crash every {np.mean(np.diff(lag2_mpc.respawn_times)):.1f} s, at progress {where}")
lag2_pursuit = race(UNIT_09, lag_ticks=2)
print(f"lag 2, Unit 9 (pure pursuit, f = 0.54): {summary(lag2_pursuit)}")
Logged, log = logged(unit11)
lag2_fallback = race(Logged, lag_ticks=2)
(lag2_agent,) = log["agent"]
print(f"lag 2, Unit 11 (MPC, then pursuit after a respawn, f = 0.54): {summary(lag2_fallback)}, "
      f"mode {lag2_agent.mode!r}")
assert not lag2_mpc.finished and lag2_mpc.collisions >= 5 and max(where) - min(where) <= 30  # a crash loop
assert clean(lag2_pursuit) and abs(lag2_pursuit.total_time - 280.00) <= TICK
assert lag2_fallback.finished and lag2_fallback.collisions == 1 and abs(lag2_fallback.total_time - 321.65) <= TICK  # U11.5
assert lag2_agent.mode == "fallback"

With one tick more lag than it models, the MPC crashes at the same corner (progress 369 to 388) about every 31 s, for as long as the race lasts: the book's tests let it run to 700 s and it never finishes (premise U11.5).
The probe record of plan `roar-010` raced it at 0.58, 0.50 and 0.40: a crash loop every time.
The MPC **trusts its lag model**: it plans as if each command acts one tick after it is sent, but the car obeys it two ticks after; every correction arrives a tick later than planned, the next plan corrects the late correction, and at speed the swings grow until the car spins.
A lower fraction does not change that.
Pure pursuit has no lag model to be wrong about: it is clean in 280.00 s on the same car.

So the fallback is a **different** controller, not the same one slowed down.
With it, this unit's file crashes once (at 30.55 s), falls back to pursuit, and finishes in **321.65 s**.
(Knowing the lag would fix the MPC on this car: in the probe record `MPC_LAG = 2` is clean on it.
But on the real car you know the lag only as well as Unit 8 measured it, and the probe record also found `MPC_LAG = 2` crashing on a lag-2 car whose throttle or steering gain was a few per cent off as well.)

**Acting on the warning.**
§5's negative result, raced on car 20: the same file at 0.58 with `WARN_FALLBACK = True` (and the line-error warning off, `WARN_ERROR_M = None`), so that |β| > 8° calls `fall_back()` at once:

In [ ]:
Logged, log = logged(variant(UNIT_11, "u11-warn", MU_FRACTION=0.58, WARN_FALLBACK=True, WARN_ERROR_M=None,
                             **NO_TIMING))
warned = race(Logged, 20)
(warned_agent,) = log["agent"]
t = warned.telemetry["t"][warned.telemetry["held"] == 0]
warned_lead = lead_time(t, np.array(log["beta"]), 8.0, np.array(log["speed"]), warned.respawn_times[0])
print(f"car 20, falling back on the beta warning: {summary(warned)}; the warning came {warned_lead:.2f} s before "
      f"the crash; limit_scale {warned_agent.limit_scale:.4f}")
print(f"car 20, falling back on the respawn alone: {summary(saved)}; limit_scale {saved_agent.limit_scale:.4f}")
print(f"the warning cost {warned.total_time - saved.total_time:.2f} s")
assert warned.collisions == 1 and abs(warned.total_time - 396.20) <= TICK  # U11.9 (car 20 still crashes)
assert abs(warned_lead - 0.75) <= TICK and abs(warned_agent.limit_scale - 0.86 ** 2) <= 1e-12

The warning came 0.75 s before the crash and the agent fell back at once, but the car spun anyway, at 99.05 s instead of 99.10 s.
Then the respawn fell back **again**, so the restart planned with 0.86² = 0.74 of the limit: 11.15 s slower than falling back on the respawn alone.
Too late to save the car, and the price is paid on the whole restart.

## 7. The submission: `agents/unit-11.py`

This unit's file is Unit 10's file with:

- **`MU_FRACTION = 0.54`**, re-tuned on the perturbed training family by the margin rule, with the shortest expected race time (§2, §4);
- **the respawn handler**, edge-triggered on the collision sensor (`RESPAWN_IMPULSE`, `REFRACTORY_TICKS`, §5);
- **the fallback**: reset the MPC's memory, re-anchor one step later, switch to pure pursuit and multiply the lateral limit by 0.86 at every respawn (§6);
- **the instruments**: β and the line error, warnings counted in `warnings` in every mode and never acted on (`WARN_FALLBACK = False`, §5).

With no respawn, `limit_scale` stays 1.0 and the handler never runs, so the file races **exactly** like `agents/unit-10.py` at the same fraction.
You have already seen it: §5's car 1 race was this file at 0.58, and the hook's car 1 race was Unit 10's:

In [ ]:
a, b = logs[1][0], hook[1]
same = a.total_time == b.total_time and all(np.sum(a.telemetry[name] != b.telemetry[name]) == 0
                                            for name in a.telemetry.dtype.names)
print(f"Unit 11 at 0.58 on car 1: {a.total_time:.2f} s; Unit 10: {b.total_time:.2f} s; every telemetry row equal: {same}")
assert same  # U11.7

The same race, row for row.
The book's tests check it on the unperturbed car and three perturbed ones (premise U11.7: 279.65 s for both on the unperturbed car), and the milestone re-races it.

| on the perturbed family (the book's tests, 24 cars each) | Unit 10 (0.58) | Unit 11 (0.54, fallback) |
|---|---|---|
| training cars clean | 21 / 24 (two crash loops) | 24 / 24 |
| held-out cars clean | 20 / 24 without a crash (inferred: Unit 11's 0.58 races are Unit 10's until the first crash) | **24 / 24** (median 279.95 s) |
| expected race time, training (DNF at 700 s) | 318.66 s | 279.95 s |
| at 0.58 with the fallback | — | every race finishes, at most 1 collision |
| the lag-2 car | never finishes | 321.65 s, 1 collision |

And what it does **not** do: on `test-circuit-seed2026` its 0.54 crashes on 4 of 12 perturbed cars (§4), and its warnings do not prevent crashes (§5).

**What this unit adds to the toolbox:**

- a crash restarts the race, so a crash at time t costs **t + 1 s**, to within a second, and a crash loop is a DNF;
- the number to minimise when the car is uncertain is the **expected race time over a family**, and a crash costs about a hundred times what one step of fraction buys;
- a seeded **family** of perturbed cars, its ranges a stated model of model error; tune on the training cars, check on the **held-out** ones, and do not expect the margin to travel to another track;
- **detection** from the sensors you have: an edge-triggered respawn signal, and warnings with measured lead times and false-alarm counts;
- a **fallback** that resets, re-anchors and degrades to a controller that does not depend on the model that failed;
- and an honest negative result, kept: a warning that does not help is an instrument, not a trigger.

Next: **Checkpoint B**, a full agent under parameter uncertainty, on perturbed cars and tracks it was not tuned on.